In [10]:
import pandas as pd
import numpy as np
import ast

In [ ]:
# Import the data 
recipe_RAW = pd.read_csv("recipe/RAW_recipes.csv")

In [11]:
# Calcul de la moyenne du nombre de mots par étape
def calculate_avg_words_per_step(steps_text):
    """Calcule la moyenne du nombre de mots par étape"""
    try:
        # Convertir la chaîne en liste si nécessaire
        if isinstance(steps_text, str):
            steps_list = ast.literal_eval(steps_text)
        else:
            steps_list = steps_text
            
        # Compter les mots pour chaque étape
        word_counts = [len(step.split()) for step in steps_list]
        
        # Retourner la moyenne
        return np.mean(word_counts) if word_counts else 0
    except:
        return 0

In [ ]:
# Garder les colonnes d'intérêt pour calculer le score d'effort culinaire
cols_of_interest = ['id', 'n_ingredients', 'ingredients', 'n_steps', 'minutes', 'steps']
# Sauvegarder le dataset nettoyé
recipe = recipe_RAW[cols_of_interest]
# Retirer les recettes avec un temps de préparation de 0 minutes ou supérieur à 6 heures
recipe = recipe[(recipe['minutes'] > 0) & (recipe['minutes'] <= 360)]
recipe.reset_index(drop=True, inplace=True)
# Ajouter la variable log_minutes au dataset
recipe['log_minutes'] = np.log1p(recipe['minutes'])
# Supprimer la recette avec 0 n_steps
recipe = recipe[recipe['n_steps'] != 0]
# Ajouter la variable avg_words_per_step au dataset
recipe['avg_words_per_step'] = recipe['steps'].apply(calculate_avg_words_per_step)
# Ajouter la variable log_n_ingredients au dataset
recipe = recipe.assign(log_n_ingredients = np.log1p(recipe['n_ingredients']))
# Sauvegarder le dataset nettoyé
recipe.to_csv("recipe/recipes_filtered.csv", index=False)

In [13]:
# Charger le nouveau dataset
recipe = pd.read_csv("recipe/recipes_filtered.csv")
print(recipe.columns)
print(recipe.shape)

Index(['id', 'n_ingredients', 'ingredients', 'n_steps', 'minutes', 'steps',
       'log_minutes', 'avg_words_per_step', 'log_n_ingredients'],
      dtype='object')
(221813, 9)


Ajouter les variables catégorielles

In [14]:
recipe = pd.read_csv("recipe/recipes_filtered.csv")

In [15]:
def categorize_prep_time(minutes):
    if minutes < 30:
        return 'Rapide'
    elif 30 <= minutes <= 90:
        return 'Moyenne'
    else:
        return 'Longue'

In [16]:
def categorize_step_length(avg_words):
    if avg_words < 10:
        return 'Simple'
    elif 10 <= avg_words < 20:
        return 'Moyennement long'
    else:
        return 'Long'

In [17]:
def categorize_n_ingredients(n_ingredients):
    if n_ingredients <= 5:
        return 'Petit'
    elif 6 <= n_ingredients <= 10:
        return 'Moyen'
    else:
        return 'Grand'

In [18]:
# Appliquer les catégorisations aux colonnes appropriées
recipe['category_minutes'] = recipe['minutes'].apply(categorize_prep_time)
recipe['category_step_length'] = recipe['avg_words_per_step'].apply(categorize_step_length)
recipe['category_n_ingredient'] = recipe['n_ingredients'].apply(categorize_n_ingredients)

# Sauvegarder le dataset avec les nouvelles variables catégorielles
recipe.to_csv("recipe/recipes_filtered_category.csv", index=False)
print(recipe.columns)
print(recipe.shape)

Index(['id', 'n_ingredients', 'ingredients', 'n_steps', 'minutes', 'steps',
       'log_minutes', 'avg_words_per_step', 'log_n_ingredients',
       'category_minutes', 'category_step_length', 'category_n_ingredient'],
      dtype='object')
(221813, 12)
